# PIDA-RL Diabetes — 16. Publicación permanente en Streamlit Community Cloud

Streamlit Community Cloud despliega la app directamente desde GitHub e inicia sesión con la misma cuenta de GitHub. No necesita tokens en este cuaderno: el código y los resultados agregados ya están en la rama `main`.

El cuaderno hace tres cosas:

1. Verifica que `main` tenga todo lo que Streamlit Cloud necesita y que no publique datos sensibles.
2. Ejecuta una prueba de humo con las mismas dependencias de `dashboard_streamlit/requirements.txt`.
3. Muestra los valores exactos para el formulario de despliegue y comprueba la URL una vez publicada.

Streamlit Cloud ejecuta `streamlit run` desde la raíz del repositorio y busca `requirements.txt` primero en la carpeta del archivo principal, por eso se usa `dashboard_streamlit/requirements.txt`.

## 1. Inicialización

In [ ]:
# === Inicialización: clona o actualiza el repositorio en la rama main ===
from pathlib import Path
import importlib.util, os, subprocess, sys

REPO_URL = 'https://github.com/JssSalas/pida-rl-datos-publicos.git'
BRANCH = os.environ.get('PIDA_BRANCH', 'main')
IN_COLAB = importlib.util.find_spec('google.colab') is not None

# En Colab el código vive en /content; fuera de Colab se usa PIDA_REPO_ROOT o la carpeta actual.
if IN_COLAB:
    REPO_ROOT = Path('/content/pida-rl-datos-publicos')
else:
    REPO_ROOT = Path(os.environ.get('PIDA_REPO_ROOT', Path.cwd() / 'pida-rl-datos-publicos')).resolve()

if (REPO_ROOT / '.git').exists():
    subprocess.run(['git', '-C', str(REPO_ROOT), 'fetch', '-q', 'origin', BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO_ROOT), 'checkout', '-q', BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO_ROOT), 'reset', '-q', '--hard', f'origin/{BRANCH}'], check=True)
else:
    subprocess.run(['git', 'clone', '-q', '--branch', BRANCH, REPO_URL, str(REPO_ROOT)], check=True)

COMMIT = subprocess.run(['git', '-C', str(REPO_ROOT), 'rev-parse', '--short', 'HEAD'],
                        text=True, capture_output=True, check=True).stdout.strip()
APP_PATH = REPO_ROOT / 'dashboard_streamlit' / 'app.py'
DATA_DIR = REPO_ROOT / 'dashboard_data' / 'v2'
REQS = REPO_ROOT / 'dashboard_streamlit' / 'requirements.txt'
print(f'Colab: {IN_COLAB} | rama: {BRANCH} | commit: {COMMIT}')
print('Repositorio:', REPO_ROOT)


## 2. Dependencias del dashboard

In [ ]:
# Instala solo las dependencias del dashboard (las mismas que usará Streamlit Cloud).
pip = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(REQS)], text=True, capture_output=True)
if pip.returncode != 0:
    faltan = [m for m in ('streamlit', 'pandas', 'plotly') if importlib.util.find_spec(m) is None]
    if faltan:
        raise RuntimeError(f'pip falló y faltan {faltan}:\n{pip.stderr[-1500:]}')
    print('pip no disponible en este entorno; se usan las dependencias ya instaladas.')
import streamlit, pandas, plotly
print('streamlit', streamlit.__version__, '| pandas', pandas.__version__, '| plotly', plotly.__version__)


## 3. Verificación del repositorio y prueba de humo

In [ ]:
# Verifica que el repositorio tenga todo lo que necesita el dashboard y que no publique datos sensibles.
import csv, py_compile

REQUERIDOS = [
    'dashboard_streamlit/app.py', 'dashboard_streamlit/data_loader.py', 'dashboard_streamlit/requirements.txt',
    'dashboard_data/v2/manifest.csv', 'dashboard_data/v2/resumen_algoritmos.csv',
    'dashboard_data/v2/rendimiento_ic.csv', 'dashboard_data/v2/criterios_pida.csv',
]
faltan = [p for p in REQUERIDOS if not (REPO_ROOT / p).exists()]
assert not faltan, f'Faltan archivos en la rama {BRANCH}: {faltan}'
py_compile.compile(str(APP_PATH), doraise=True)

versionados = subprocess.run(['git', '-C', str(REPO_ROOT), 'ls-files'], text=True,
                             capture_output=True, check=True).stdout.splitlines()
PROHIBIDOS = ('.sav', '.dta', '.parquet', 'secrets.toml', '.env', 'cohorte_diabetes.csv')
sensibles = [f for f in versionados if f.endswith(PROHIBIDOS)]
assert not sensibles, f'Archivos que no deben publicarse: {sensibles}'

# Ningún CSV publicado debe traer identificadores de perfil o de vivienda.
COLUMNAS_PROHIBIDAS = {'FOLIO_INT', 'folio', 'perfil_id', 'id_perfil'}
for f in versionados:
    if f.startswith('dashboard_data/') and f.endswith('.csv'):
        with open(REPO_ROOT / f, newline='', encoding='utf-8') as fh:
            encabezado = set(next(csv.reader(fh)))
        assert not (encabezado & COLUMNAS_PROHIBIDAS), f'{f} contiene identificadores: {encabezado & COLUMNAS_PROHIBIDAS}'
print(f'Verificación correcta: {len(REQUERIDOS)} archivos requeridos, sin datos crudos ni identificadores.')


In [ ]:
# Prueba de humo: recorre todas las secciones de la barra lateral sin servidor ni túnel.
import warnings
from streamlit.testing.v1 import AppTest

os.environ['PIDA_DASHBOARD_DATA_DIR'] = str(DATA_DIR)
warnings.filterwarnings('ignore')
at = AppTest.from_file(str(APP_PATH), default_timeout=180).run()
secciones = list(at.sidebar.radio[0].options)
resultado = []
for seccion in secciones:
    at.sidebar.radio[0].set_value(seccion).run()
    errores = [e.value for e in at.exception] + [e.value for e in at.error]
    assert not errores, f'La sección «{seccion}» muestra errores: {errores[:2]}'
    resultado.append(f'{seccion}: {len(at.dataframe)} tablas')
print(f'Prueba de humo correcta en {len(secciones)} secciones, sin errores:')
print('\n'.join('  - ' + r for r in resultado))


## 4. Valores para el formulario de despliegue

In [ ]:
from IPython.display import Markdown, display

CONFIG = {
    'Repository': 'JssSalas/pida-rl-datos-publicos',
    'Branch': BRANCH,
    'Main file path': 'dashboard_streamlit/app.py',
    'App URL (sugerida)': 'pida-rl-diabetes',
    'Python version (Advanced settings)': '3.12',
    'Secrets (Advanced settings)': 'ninguno',
}
filas = '\n'.join(f'| {k} | `{v}` |' for k, v in CONFIG.items())
display(Markdown(f"""| Campo | Valor |
|---|---|
{filas}

**Pasos**

1. Abre [share.streamlit.io](https://share.streamlit.io/) y elige **Continue with GitHub** con la cuenta `JssSalas`.
2. Si es la primera vez, autoriza a Streamlit para leer tus repositorios.
3. Pulsa **Create app → Deploy a public app from GitHub** y llena los campos de la tabla.
4. En **Advanced settings** selecciona Python 3.12 (la versión no se puede cambiar después sin borrar la app).
5. Pulsa **Deploy** y espera a que termine la instalación (2 a 5 minutos).
6. Copia la URL final en la celda siguiente para comprobarla.

Cada `git push` a `{BRANCH}` actualiza la app automáticamente. Commit verificado en este cuaderno: `{COMMIT}`.
"""))


## 5. Comprobar la app publicada

Pega la URL que asignó Streamlit Cloud. Una app inactiva puede tardar en despertar; la celda reintenta durante unos minutos.

In [ ]:
import time, urllib.request, urllib.error

STREAMLIT_APP_URL = ''   # ejemplo: 'https://pida-rl-diabetes.streamlit.app'

if not STREAMLIT_APP_URL:
    print('Escribe la URL de la app en STREAMLIT_APP_URL y vuelve a ejecutar esta celda.')
else:
    salud = STREAMLIT_APP_URL.rstrip('/') + '/_stcore/health'
    for intento in range(10):
        try:
            with urllib.request.urlopen(salud, timeout=30) as r:
                print('App disponible:', r.status, r.read().decode()[:20], '|', STREAMLIT_APP_URL)
                break
        except (urllib.error.URLError, TimeoutError) as error:
            print(f'Intento {intento + 1}: aún no responde ({error}); reintento en 20 s.')
            time.sleep(20)
    else:
        print('La app no respondió. Revisa los registros en share.streamlit.io → Manage app.')


## 6. Registro para el reporte

Guarda la URL y el commit desplegado como evidencia del entregable 4.4 (despliegue en GitHub y Streamlit Cloud).

In [ ]:
import json
from datetime import datetime, timezone

registro = {
    'fecha_utc': datetime.now(timezone.utc).isoformat(timespec='seconds'),
    'repositorio': CONFIG['Repository'], 'rama': BRANCH, 'commit': COMMIT,
    'archivo_principal': CONFIG['Main file path'], 'url': STREAMLIT_APP_URL or None,
}
print(json.dumps(registro, ensure_ascii=False, indent=2))
